# Silver Transform
Test the Silver layer transformation independently.
Note: You should have run bronze_ingestion first so there is pending Bronze data.

In [ ]:
from datetime import datetime, timezone
import polars as pl
from electricity_maps.layers.silver import transform_silver_flows, transform_silver_mix
from electricity_maps.utils.state import PipelineState
from electricity_maps.config import get_settings

In [ ]:
settings = get_settings()
state = PipelineState(settings.data_dir)

now = datetime.now(timezone.utc)
silver_ts = int(now.timestamp())

In [ ]:
ts_list = state.pickup_ready("bronze")
if not ts_list:
    print("No pending bronze data found. Run 02_bronze_ingestion first.")
else:
    print(f"Picked up Bronze batches: {ts_list}")
    
    transform_silver_mix(ts_list)
    transform_silver_flows(ts_list)
    
    # Mark state
    state.mark_complete("bronze", ts_list)
    state.init_layer("silver", silver_ts, now)
    state.mark_ready("silver", silver_ts, now, len(ts_list))
    
    print("Silver transformation completed!")

In [ ]:
# Let's inspect the Silver Delta tables
print("Silver Mix Table:")
try:
    df_mix = pl.read_delta(str(settings.silver_dir / "electricity_mix"))
    display(df_mix.head())
except Exception as e:
    print("Error reading silver mix:", e)

In [ ]:
print("\nSilver Flows Table:")
try:
    df_flows = pl.read_delta(str(settings.silver_dir / "electricity_flows"))
    display(df_flows.head())
except Exception as e:
    print("Error reading silver flows:", e)